# step B — directive-token (지침 속 'camelCase' 토큰을 보나?) · 모델 패밀리

**대응 RQ:** RQ3 관측(지침 쪽) — RQ2는 코드 쪽, 이건 지침 어텐션을 stepB 방법으로 봄(인과는 step4).

**무엇을 하나** — 지침 안의 **표기 지시어 토큰**('camelCase')을 별도 span으로 잡아 **토큰당 어텐션**을 코드 토큰·지침 전체와 층별로 비교. Qwen에서 '지시어는 코드(L25)보다 늦은 L27에서 피크'가 나왔다. **다른 패밀리(DeepSeek·Llama)에서도 이 '층 순서' 패턴이 재현되나?**

**모델 무관 하네스** — `family`는 라벨, GQA·어텐션·KV는 config에서 자동. 셀3에서 모델만 바꿔 돌린다. 층 수가 달라 **피크 층은 각 모델 자기 층에서 자동 탐색**(L25/L27 하드코딩 안 함).

> **메모리:** `output_attentions`라 무겁다 → **소형(≤3B) 권장**(T4). 7B는 T4에서 OOM. eager 필수. 재개 가능.

In [ ]:
# 환경 설정
!pip install -q transformers accelerate torch matplotlib pandas numpy
import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')
SEED=0; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin stepB/directive-token
!git checkout stepB/directive-token
!git pull --quiet origin stepB/directive-token
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# 모델 선택 — 하나만 활성화하고 실행 (한 번에 한 모델)
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation)
from harness.tasks import NAME_PAIR_POOL

# --- 여기서 모델 선택 (T4 안전 = 소형) ---
# MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')      # 이미 함
MODEL = ModelSpec(name='deepseek-ai/deepseek-coder-1.3b-instruct', family='deepseek', dtype='float16')
# MODEL = ModelSpec(name='unsloth/Llama-3.2-3B-Instruct', family='llama', dtype='float16')
# 큰 GPU면: deepseek-ai/deepseek-coder-6.7b-instruct / codellama/CodeLlama-7b-Instruct-hf

FAMILY = MODEL.family
STEP = f'stepB_directive_{FAMILY}'                # 모델별 결과 분리
N_FUNCTIONS = 12
N_BLOCKS_USE = 10
BLOCKS = list(range(N_BLOCKS_USE))
SEEDS = [0]
FLIP  = [(Notation.CAMEL, 6), (Notation.SNAKE, 6)]
SWEEP = [(Notation.CAMEL, n) for n in (4, 3, 2, 1, 0)]
SPECS = list(dict.fromkeys(FLIP + SWEEP))

def make(target, n, b, s):
    return Condition(model=MODEL,
        preceding=PrecedingCode(n_compliant=n, n_functions=N_FUNCTIONS, composition=Composition.POOL, pool_block=b),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=target), seed=s)

conditions = [make(t, n, b, s) for (t, n) in SPECS for b in BLOCKS for s in SEEDS]
PREDICTION = ('지시어 토큰(camelCase)이 코드 형태 토큰보다 늦은 층에서 피크(Qwen L27 vs L25)이면 '
              '층 순서 패턴이 패밀리 일반화.')
print(f'모델: {MODEL.name} (family={FAMILY}) | STEP={STEP}')
print(f'{len(conditions)} 조건 = {len(SPECS)} spec x {len(BLOCKS)} block')

In [ ]:
# 실행 — observe(지시어 span 자동 포함). 재개.
from harness import run, ResultRecord, save_result, result_path
from harness.results import load_result
from harness.model import load_model

handle = load_model(MODEL, attn_implementation='eager')   # 어텐션 가중치 필수
print('layers:', handle.num_layers, '| GQA:', handle.gqa_info())
new = skipped = 0
for i, c in enumerate(conditions, 1):
    p = result_path(c, step=STEP)
    if p.exists(): skipped += 1
    else:
        out = run(c, handle=handle, mode='observe')
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics, step=STEP, rq='RQ3', prediction=PREDICTION))
        new += 1
    if i % 10 == 0 or i == len(conditions): print(f'[{i}/{len(conditions)}] 새 {new} / 건너뜀 {skipped}')
print('완료.')

In [ ]:
# 결과 로드
from harness import result_path
from harness.results import load_result
records = [load_result(result_path(c, step=STEP)) for c in conditions]
print('로드:', len(records), '-> results/'+STEP+'/')

In [ ]:
# 요약 — 피크 층 자동 탐색: 지시어가 코드보다 늦게 피크하나? (모델 무관)
import numpy as np, matplotlib.pyplot as plt
c6 = [r for r in records if r.condition.instruction.target_notation.value=='camel' and r.condition.preceding.n_compliant==6]
NL = max(int(L) for r in records for L in r.metrics.per_layer)+1
def pertok(span, L):
    v=[]
    for r in c6:
        pl=r.metrics.per_layer.get(str(L), r.metrics.per_layer.get(L,{}))
        a=pl.get(f'{span}__attention_weight'); n=r.metrics.extra['span_token_counts'].get(span)
        if a is not None and n: v.append(a/n)
    return float(np.mean(v)) if v else np.nan
def curve(span): return [pertok(span,L) for L in range(NL)]

spans={'instr_target_word':'directive "camelCase"','code_camel':'code camel tok',
       'code_snake':'code snake tok','instruction':'instruction (whole)'}
print(f'=== 모델 {FAMILY} ({NL}층): per-token 피크 층 ===')
peaks={}
for sp,lab in spans.items():
    c=curve(sp); pk=int(np.nanargmax(c)); peaks[sp]=(pk,c[pk])
    print(f'  {lab:24} 피크 L{pk:<2} ({c[pk]:.5f})  상대 {pk/NL:.2f}')
dpk=peaks['instr_target_word'][0]; cpk=peaks['code_camel'][0]
print(f"\n>> 지시어 피크 L{dpk} vs 코드 피크 L{cpk} : "
      + ('지시어가 늦음(Qwen 패턴 재현)' if dpk>cpk else '지시어가 같거나 빠름(패턴 다름)'))

layers=list(range(NL))
plt.rcParams.update({'font.size':10,'axes.grid':True,'grid.alpha':.3})
fig,ax=plt.subplots(figsize=(8,4.2))
for sp,lab,col in [('instr_target_word','directive "camelCase"','#B0392B'),
                   ('code_camel','code camel tok','#2E7D52'),('instruction','instruction (whole)','#7B3FA0')]:
    ax.plot(layers,curve(sp),marker='.',ms=4,label=lab,color=col)
ax.axvline(cpk,color='#2E7D52',ls=':',lw=1); ax.axvline(dpk,color='#B0392B',ls=':',lw=1)
ax.set_xlabel('layer'); ax.set_ylabel('attention PER token'); ax.set_title(f'{FAMILY}: directive vs code peak layer')
ax.legend(fontsize=8,loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=3,frameon=False)
plt.tight_layout(rect=[0,0.06,1,1]); plt.savefig(f'stepB_directive_{FAMILY}.png',dpi=120); plt.show()

In [ ]:
# 결과 다운로드
import shutil
shutil.make_archive(f'stepB_directive_{FAMILY}_results', 'zip', 'results/'+STEP)
try:
    from google.colab import files; files.download(f'stepB_directive_{FAMILY}_results.zip')
except Exception as e:
    print('Colab 아님(수동):', f'stepB_directive_{FAMILY}_results.zip', e)